In [1]:
!pip install -q transformers accelerate peft trl bitsandbytes datasets numpy


In [ ]:
!pip uninstall -y trl numpy
!pip install trl==0.7.4 numpy

Found existing installation: trl 0.7.4
Uninstalling trl-0.7.4:
  Successfully uninstalled trl-0.7.4
Found existing installation: numpy 2.3.2
Uninstalling numpy-2.3.2:
  Successfully uninstalled numpy-2.3.2
  Using cached trl-0.7.4-py3-none-any.whl.metadata (10 kB)
  Using cached numpy-2.3.2-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
Using cached trl-0.7.4-py3-none-any.whl (133 kB)
Using cached numpy-2.3.2-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.9 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 4.1.0 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.38.2 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.2 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
import torch

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)

dataset = load_dataset("mlabonne/guanaco-llama2-1k", split="train")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
You are calling `save_pretrained` to a 4-bit converted model, but your `bitsandbytes` version doesn't support it. If you want to save 4-bit models, make sure to have `bitsandbytes>=0.41.3` installed.

In [7]:
def formatting_func(example):
    # The dataset examples have 'text', 'instruction', 'input', and 'output' keys.
    # We will construct the prompt based on these keys as per the Alpaca format.
    if example["input"]:
        return f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}"
    else:
        return f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"

from transformers import DataCollatorForLanguageModeling, TrainingArguments
from trl import SFTTrainer

# Remove the manual dataset tokenization as SFTTrainer will handle it.
# tokenized_dataset = dataset.map(
#     lambda x: tokenizer(x["formatted"], truncation=True, padding="max_length", max_length=512),
#     remove_columns=dataset.column_names,
#     batched=True,
# )

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [13]:
training_args = TrainingArguments(
    output_dir="./tinyllama-guanaco",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=50,
    fp16=False,
    bf16=False,  # Only if supported
    push_to_hub=False
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,  # Use the original dataset
    tokenizer=tokenizer,
    data_collator=data_collator,
    args=training_args,
    formatting_func=formatting_func,
    dataset_text_field="text", # Specify the text field for SFTTrainer
    packing=False,
)
import os
os.environ["WANDB_DISABLED"] = "true"

trainer.train()

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/usr/local/lib/python3.11/dist-packages/trl/trainer/sft_trainer.py:194: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Step,Training Loss
10,1.552900
20,1.599200
30,1.633700
40,1.669000
50,1.661900
60,1.574500
70,1.599700
80,1.609600
90,1.617600
100,1.678400


Checkpoint destination directory ./tinyllama-guanaco/checkpoint-50 already exists and is non-empty. Saving will proceed but saved results may be invalid.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Checkpoint destination directory ./tinyllama-guanaco/checkpoint-100 already exists and is non-empty. Saving will proceed but saved results may be invalid.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


TrainOutput(global_step=125, training_loss=1.6076491508483888, metrics={'train_runtime': 1167.023, 'train_samples_per_second': 0.857, 'train_steps_per_second': 0.107, 'total_flos': 3574855237607424.0, 'train_loss': 1.6076491508483888, 'epoch': 1.0})

In [10]:
trainer.save_model("tinyllama-finetuned")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [15]:
trainer.push_to_hub()


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Uploading...:   0%|          | 0.00/5.04M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Rahul31/tinyllama-guanaco/commit/df764619e208a2b1eb70f128ef60c3345dfa6c9d', commit_message='End of training', commit_description='', oid='df764619e208a2b1eb70f128ef60c3345dfa6c9d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Rahul31/tinyllama-guanaco', endpoint='https://huggingface.co', repo_type='model', repo_id='Rahul31/tinyllama-guanaco'), pr_revision=None, pr_num=None)

In [20]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("Rahul31/tinyllama-guanaco")
tokenizer = AutoTokenizer.from_pretrained("Rahul31/tinyllama-guanaco")

input_text = "Hi, how are you"
inputs = tokenizer(input_text, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Hi, how are you? I'm doing well. I hope you are too. I'm glad to hear that. I'm doing well too. I hope you are too. I'm doing well too. I hope you are too. I'm


In [16]:
trainer.save_model("tinyllama-finetuned")
tokenizer.save_pretrained("tinyllama-finetuned")


('tinyllama-finetuned/tokenizer_config.json',
 'tinyllama-finetuned/special_tokens_map.json',
 'tinyllama-finetuned/tokenizer.model',
 'tinyllama-finetuned/added_tokens.json',
 'tinyllama-finetuned/tokenizer.json')